[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xylem-lab/GEO-ANOM-dashboard/blob/research/accuracy-assessment-2026-11-05/notebooks/14_accuracy_assessment.ipynb)

# 14 — Verification v1.1 of map v1: accuracy assessment by design

**Question.** How accurate is **map v1** (release_v1: 2,833 poultry houses, 6.98 km² roof)? Measured with a random sample, a written protocol and two independent interpreters: precision, recall, houses standing and total roof, each with a 95% confidence interval. Replaces the single-reviewer 97% precision and the informal 21/24 check.

**Protocol:** [`docs/TASK1_ACCURACY_PROTOCOL.md`](../docs/TASK1_ACCURACY_PROTOCOL.md), verification v1.1, **revision 2** (frozen 2026-11-10). Pilot 1 failed (κ 0.58), so the labels were revised and the calibration and pilot photos replaced. Run sheet: [`docs/NB14_RUN_SHEET.md`](../docs/NB14_RUN_SHEET.md).

**Interpreters** (Umesh Adari `UA`, Monica `Monica`): run the steps **1–8** in order, then stop. Everything under **Coordinator only** is for UA as coordinator; interpreters don't run it.

**Inputs.** The saved samples in `Task1_results/accuracy_v1/` (shared drive), NAIP 2023 photos streamed from Planetary Computer (never stored). **Outputs** (shared drive only, they name places): `labels/labels_<interpreter>.csv`, and from the coordinator steps `adjudicated.csv`, `estimates.csv`, `strata.csv`, `residual.csv`, `agreement.csv`, `run_meta.json`.

**Time.** Loading: ~1 min. Labelling, at about 10 s a photo: calibration + pilot ~20 min; the rest ~3.5 h of buildings and ~6 h of squares per interpreter, in as many sittings as you like.

In [ ]:
# ── Setup: the same cell works in Google Colab and locally ──────────────────────
import os, sys, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "research/accuracy-assessment-2026-11-05"   # branch, tag or commit to run; the exact commit is recorded in run_meta.json

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                     # results go to the shared drive's Task1_results
    ROOT = Path("/content/GEO-ANOM-dashboard")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/xylem-lab/GEO-ANOM-dashboard", str(ROOT)], check=True)
    # always bring the copy up to date (a reused Colab session keeps an old copy)
    head = lambda: subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout
    before = head()
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=True)
    remote = f"origin/{CODE_VERSION}"
    target = remote if subprocess.run(["git", "-C", str(ROOT), "rev-parse", "-q", "--verify", remote],
                                      capture_output=True).returncode == 0 else CODE_VERSION
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", target], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
    if head() != before and "geo_anom" in sys.modules:   # new code, but the old version is already loaded
        raise RuntimeError("Code updated. Runtime -> Restart session, then Run all again.")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
# reload edited geo_anom/ code automatically, without restarting the kernel
if "google.colab" not in sys.modules:   # Colab's IPython can't load autoreload on Python 3.12+ (no `imp`)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

from geo_anom.task1 import env, region
from geo_anom.task1 import county as C
from geo_anom.task1 import accuracy as A
from geo_anom.task1.evaluate import load_ground_truth, GROUND_TRUTH
print("running in", "Colab" if IN_COLAB else "local", "| results ->", env.results_root())

## 1. Set your name

Exactly `"UA"` or `"Monica"`. With it empty, every labelling screen is skipped (so Run All always works).

In [ ]:
INTERPRETER = ""                 # who is labelling now: exactly "UA" or "Monica"
INTERPRETERS = ["UA", "Monica"]  # both interpreters (for the comparisons)
ACC = A.acc_dir()                # Task1_results/accuracy_v1 on the shared drive
LABELS_DIR = ACC / "labels"
store = A.Store(LABELS_DIR, INTERPRETER) if INTERPRETER else None
frame_all, metas = A.load_samples(ACC)
R2 = [r for r in INTERPRETERS if r]
if frame_all is None:
    print("round 2 isn't drawn yet: the coordinator runs step C3 first")
else:
    pilot_keys = frame_all[frame_all.pilot & (frame_all.role == "sample")].key
    print(f"samples loaded | protocol {A.sha256_file(A.PROTOCOL)[:12]} ({A.PROTOCOL_LABEL}) | labels -> {LABELS_DIR}")
    print("calibration", (frame_all.role == "calibration").sum(), "| pilot", len(pilot_keys),
          "| sampled buildings", ((frame_all.role == "sample") & (frame_all.kind == "object")).sum(),
          "| sampled squares", ((frame_all.role == "sample") & (frame_all.kind == "cell")).sum())

## 2. How to decide

**Slow down.** About 10 seconds a photo is normal. Look at the close-up, and write a note whenever you tick *check later*. (In round 1 we averaged 1.5–2.6 s a photo, and the pilot failed.)

**The question:** *what is at the pink crosshair on the 2023 photo?* Judge the building the crosshair sits on, or the nearest one if it's between two.

**A standing poultry house has all three:**
1. **Shape:** long, narrow, single-storey; about 12–20 m wide, 100–200 m long, at least 5× longer than wide (use the scale bar).
2. **Roof complete** on the 2023 photo.
3. **A poultry cue:** fans or doors on an end, feed bins along a side, identical houses side by side, a manure shed or composter nearby.

Idle or rusty houses count. Turkey and layer houses count.

**Pick one label, always:**

| Label | Use when |
|---|---|
| **house** | All three points above |
| **gone** | A house was clearly here: bare slab, scar or roofless frame |
| **under construction** | Frame or unfinished roof |
| **not a house** | Anything else, including look-alikes: equipment sheds, greenhouses / hoop houses, manure sheds, long barns with no poultry cues, dairy barns (wider, open sides, silos, lagoon), warehouses, roads, fields |
| **can't tell: photo unreadable** | The photo doesn't show the ground at the crosshair: cloud or haze, heavy blur, black or cut-off tile edge, no 2023 photo loaded |

**If you can see the building but can't decide what it is, "can't tell" is the wrong label.** Pick the label you think is most likely and tick **check later**.

- **Check later:** not sure, or want this photo looked at again. Your label still counts. Say why in the note.
- **Poultry, other form:** with *not a house*, for a poultry building outside the shape rule (a wide or multi-storey layer house, a short duck house).
- **Don't** talk about labels with the other interpreter until you've both finished a phase, and don't look items up in our map or the KMZ. **← back** fixes a wrong click; your last label counts.

**Examples** (photos from round 1, not in this round; the label under each is the agreed answer):

In [ ]:
if frame_all is not None:
    A.show_examples(ACC / "guide_examples.csv", frame_all)

## 3. Calibration (30 photos)

Label all 30 on your own. These photos are never part of the result.

In [ ]:
if store is not None and frame_all is not None:
    A.label_ui(A.queue(frame_all, INTERPRETER, "calibration"), store, "calibration")
else:
    print("INTERPRETER is empty: labelling skipped")

## 4. Compare and discuss (together, after both finished calibration)

Shows how often you agreed and every photo where you differ or one of you ticked *check later*. Talk each one through using the guide in step 2. If the guide's wording is the problem, tell the coordinator **before** the pilot (the protocol gets a new revision).

In [ ]:
labels = A.load_labels(LABELS_DIR)
def compare(phase):
    L = labels[labels.phase == phase]
    if L.empty or not set(R2) <= set(L.reviewer):
        print(f"{phase}: needs both interpreters' labels"); return None
    w = A.wide(L, R2).dropna()
    fl = A.flagged(L, R2).reindex(w.index).fillna(False)
    notes = L.pivot(index="key", columns="reviewer", values="note").reindex(w.index).fillna("")
    out = w.assign(**{"check later": fl}, **{f"note {r}": notes[r] for r in R2})
    print(f"{phase}: same label on {(w[R2[0]] == w[R2[1]]).mean():.0%} of {len(w)}, "
          f"house / not house the same on {(A.binary(w[R2[0]]).values == A.binary(w[R2[1]]).values).mean():.0%}")
    return out
cal = compare("calibration")
if cal is not None:
    display(cal[(cal[R2[0]] != cal[R2[1]]) | cal["check later"]].reset_index(drop=True))
    display(A.pace(A.load_raw(LABELS_DIR)))

## 5. Pilot (74 photos)

On your own, after the discussion.

In [ ]:
if store is not None and frame_all is not None:
    A.label_ui(A.queue(frame_all, INTERPRETER, "pilot"), store, "pilot")

## 6. Gate (after both finished the pilot)

**PASS** needs Cohen's kappa (house / not house) ≥ 0.8 on all 74 photos and *can't tell* on at most 5% of them. **FAIL** or **STOP** → don't go on; the coordinator revises the protocol. Never relabel the same pilot photos.

In [ ]:
labels = A.load_labels(LABELS_DIR)
if frame_all is not None and set(R2) <= set(labels.reviewer):
    g = A.gate(labels, frame_all, R2, pilot_keys)
    print(f"pilot: {g['n']} of {g['of']} labelled by both")
    if g["n"]:
        ct = g["can't tell share"]
        print(f"kappa {g['kappa']:.2f} (95% CI {g['kappa low']:.2f}-{g['kappa high']:.2f}), "
              f"same house / not {g['same (house / not)']:.0%}, can't tell {ct:.0%}")
        display(A.agreement(labels, frame_all, R2, keys=pilot_keys).round(3))
        display(A.agreement_by_flag(labels, R2, keys=pilot_keys).round(3))
    print("->", g["verdict"])
else:
    print("gate: needs both interpreters' pilot labels")

## 7. Main labelling (only after the gate says PASS)

7a: the rest of the first sample (~380 photos). 7b: the addendum buildings (western detections, footprint candidates, published-model outlines; ~750 photos). Same photos and labels.

In [ ]:
if store is not None and frame_all is not None:
    A.label_ui(A.queue(frame_all, INTERPRETER, "main"), store, "main")

In [ ]:
if store is not None and frame_all is not None:
    A.label_ui(A.queue(frame_all, INTERPRETER, "addendum objects"), store, "addendum objects")

## 8. Squares (after 7)

The dashed square is 1 km; every building we already know is outlined in cyan. **Count the standing poultry houses with their centre inside the square and no outline.** Most squares are 0. In the note, say where (e.g. "NE ninth, 3 houses"). 8a: Eastern Shore (400), 8b: west (149). Loading the outlines takes ~1 min.

In [ ]:
if store is not None and frame_all is not None:
    release = gpd.read_file(env.results_root() / "final" / "release_v1" / "maryland_poultry_houses.geojson")
    known = A.known_layer(release, load_ground_truth(), A.load_detections(A.west_counties()),
                          frame_all[frame_all.stratum.isin(["S5ES", "S5WR", "S5WD"])])
    A.cell_ui(A.queue(frame_all, INTERPRETER, "cells"), store, known)

In [ ]:
if store is not None and frame_all is not None:
    A.cell_ui(A.queue(frame_all, INTERPRETER, "addendum cells"), store, known, phase="addendum cells")

---
# Coordinator only (interpreters don't run these)

C1 frames, C2 the original draws (load only), C3 round 2, C4 guide examples, C5 agreement and pace, C6 adjudication, C7 estimates.

## C1. Frames: every outline and traced house in its stratum, and the 1 km cells

Must equal release_v1's own tables (2,450 / 327 / 56 / 594).

In [ ]:
SEED = 20261105
RELEASE = env.results_root() / "final" / "release_v1"
release = gpd.read_file(RELEASE / "maryland_poultry_houses.geojson")
traced = load_ground_truth()
objects, cells, K = A.frames(release, traced)
N = objects.groupby("stratum").size()
assert (N["S1"], N["S2"], N["S3"], N["S4"]) == (2450, 327, 56, 594), N
assert N[["S1", "S2", "S3"]].sum() == len(release)
display(pd.crosstab(pd.concat([objects.county, cells.county], ignore_index=True),
                    pd.concat([objects.stratum, cells.stratum], ignore_index=True), margins=True, margins_name="total"))

## C2. The original draws (loaded, never redrawn)

First sample (seed 20261105), addendum (20261106), S9 (20261107). Each file is checked against its SHA-256. Drawing code for an empty folder: `A.draw`, `A.draw_addendum`, `A.draw_amendment_a1` (see git history of this notebook before 2026-11-10 for the full cells).

In [ ]:
FRAME, META = ACC / "frame.geojson", ACC / "sample_meta.json"
frame = gpd.read_file(FRAME).to_crs(A.UTM)
sm = json.load(open(META))
assert A.sha256_file(FRAME) == sm["frame_sha256"], "frame.geojson changed since it was drawn"
sm12 = json.load(open(ACC / "sample_meta_addendum.json"))
sm_a1 = json.load(open(ACC / "sample_meta_a1.json"))
k_f, q = sm12["k_f"], pd.DataFrame(sm12["capture_rates"]).set_index("rate")
for name, m in (("first sample", sm), ("addendum", sm12), ("S9", sm_a1)):
    print(f"{name}: drawn {m['made_at'][:10]}, seed {m['seed']}, SHA-256 {m['frame_sha256'][:16]}")
s = frame[frame.role == "sample"]
display(pd.crosstab(s.county, s.stratum, margins=True, margins_name="total"))

## C3. Round 2: new calibration and pilot photos (once)

Pilot 1 failed (κ 0.58). The round-1 labels are in `labels/archive_round1/` with `pilot1_record.json`. `A.start_round2` follows the protocol §3: no new random numbers, every choice follows the seed-20261105 ranks; refuses if any new photo was labelled in round 1. Saved as `frame_r2.geojson` + `sample_meta_r2.json`; `frame.geojson` is not rewritten.

In [ ]:
ARC = LABELS_DIR / "archive_round1"
rec = json.load(open(ARC / "pilot1_record.json"))
FRAME_R2, META_R2 = ACC / "frame_r2.geojson", ACC / "sample_meta_r2.json"
if not FRAME_R2.exists():
    used = set(rec["calibration_keys"]) | set(rec["pilot_keys"])
    f2 = A.start_round2(frame, used)
    f2.to_crs(4326).to_file(FRAME_R2, driver="GeoJSON")
    json.dump(A.meta(RELEASE, GROUND_TRUTH, SEED, {
        "frame_sha256": A.sha256_file(FRAME_R2), "based_on_frame_sha256": sm["frame_sha256"], "k": sm["k"],
        "round": A.ROUND, "protocol_label": A.PROTOCOL_LABEL, "round1_record_sha256": A.sha256_file(ARC / "pilot1_record.json"),
        "round1_labels_sha256": rec["archived_sha256"], "new_random_numbers": False}), open(META_R2, "w"), indent=1, default=str)
    print("drew round 2 and saved it:", FRAME_R2)
    frame_all, metas = A.load_samples(ACC)
f2 = frame_all[frame_all.stratum.isin(A.V11_OBJECT + A.V11_CELL)]
print(f"pilot 1 (archived): kappa {rec['pilot']['kappa_house_not']:.2f} "
      f"({rec['pilot']['kappa_ci95'][0]:.2f}-{rec['pilot']['kappa_ci95'][1]:.2f}), n {rec['pilot']['n']}, {rec['date']}")
display(f2[f2.round2 != ""].groupby(["stratum", "round2"]).size().rename("items").reset_index())
display(f2.role.value_counts().rename("items").to_frame())
pilot_keys = frame_all[frame_all.pilot & (frame_all.role == "sample")].key
assert not (set(pilot_keys) | set(frame_all.key[frame_all.role == "calibration"])) & (set(rec["calibration_keys"]) | set(rec["pilot_keys"]))
print("round 2 SHA-256", metas["round 2"]["frame_sha256"][:16], "| calibration", (frame_all.role == "calibration").sum(),
      "| pilot", len(pilot_keys), "| none labelled in round 1")

## C4. Guide examples (round-1 photos)

Candidates: every round-1 disagreement plus a few agreed photos per label. The coordinator settles each one's correct label with the PIs in `guide_examples.csv` on the shared drive (`label`, `case` from `A.EXAMPLE_CASES`, a one-line `why`, `status = confirmed`). Only confirmed rows appear in step 2. The contact sheet (shared drive only) shows each candidate with both round-1 labels.

In [ ]:
EX = ACC / "guide_examples.csv"
arch = A.load_labels(ARC)
if not EX.exists():
    A.example_candidates(arch, frame_all, R2).to_csv(EX, index=False)
ex = pd.read_csv(EX, dtype=str).fillna("")
SHEET = ACC / "guide_examples_contact_sheet.png"
if not SHEET.exists():
    A.contact_sheet(ex, frame_all, SHEET)
print(f"{len(ex)} candidates, {(ex.status == 'confirmed').sum()} confirmed | contact sheet: {SHEET.name}")
display(ex.drop(columns="key").groupby(["case", "status"]).size().rename("photos").reset_index())

## C5. Agreement before adjudication, pace and the check-later flag

In [ ]:
labels = A.load_labels(LABELS_DIR)
display(A.pace(A.load_raw(LABELS_DIR)))
sample_keys = frame_all[frame_all.role == "sample"].key
agree = None
if set(R2) <= set(labels.reviewer):
    agree = A.agreement(labels, frame_all, R2, keys=sample_keys)
    display(agree.round(3))
    display(A.agreement_by_flag(labels, R2, keys=sample_keys).round(3))
    print(A.cell_agreement(labels, R2))
    agree.to_csv(ACC / "agreement.csv", index=False)
else:
    print("agreement: needs both interpreters")

## C6. Adjudication

Every disagreement, every *can't tell*, every *check later*, and the 3 round-1 pilot items kept in the sample (`A.KEPT`, label by adjudication only). Decide together and record each with `A.adjudicate(...)`: decision, reason, who decided. Still split → the third person; if they can't decide, it stays *can't tell*.

In [ ]:
ADJ = ACC / "adjudicated.csv"
adj = pd.read_csv(ADJ, dtype={"value": str}) if ADJ.exists() else None
kept = frame_all[frame_all.round2 == A.KEPT][["key", "stratum", "county"]]
if set(R2) <= set(labels.reviewer):
    adjq = A.adjudication_queue(labels[labels.key.isin(sample_keys)], R2)
    adjq = pd.concat([adjq, kept[["key"]]], ignore_index=True).merge(frame_all[["key", "stratum", "county"]], on="key")
    left = adjq[~adjq.key.isin(adj.key)] if adj is not None else adjq
    print(f"{len(adjq)} to adjudicate, {len(left)} not yet decided")
    display(left.head(30))
else:
    print(f"adjudication waits for both interpreters; {len(kept)} round-1 items kept for it")
# one call per decision:
# A.adjudicate(ADJ, "usgs:1234", "gone", "bare slab on the 2023 photo; Google Earth 2023-09 agrees", "UA+JH")

## C7. Estimates

Agreed label, else the adjudicated one, else *can't tell*. Main estimate: *can't tell* and *under construction* count as not a house; sensitivity rows count them as houses. Houses on a list (§5), residual (§6) separately. CIs: finite-population-corrected stratified bootstrap, widened to the exact one-sided bound for an all-or-nothing sampled stratum. S5WD (developed land) is an assumption check, not in the interval.

In [ ]:
step0_es = A.load_detections(C.EASTERN_SHORE)
PUBLISHED_ES = len(step0_es)
links = A.step0_links(release, step0_es)
fe = frame_all.assign(n_pub=frame_all.key.map(links).fillna(0))
fe.loc[fe.stratum == "S9", "n_pub"] = 1
print(f"published-model outlines on the Eastern Shore: {PUBLISHED_ES:,} = {int(links.sum()):,} linked + {int((fe.stratum == 'S9').sum())} in S9")
if set(R2) <= set(labels.reviewer):
    final = A.final_labels(labels[labels.key.isin(sample_keys)], R2, adj)
    labelled = fe[fe.role == "sample"].key.isin(final.index)
    print(f"sampled items with a final label: {labelled.sum()} of {len(labelled)}")
    strata = A.stratum_table(fe, final)
    display(strata)
    runs = {"main": {}, "can't tell = house": {"cant_tell_as_house": True},
            "under construction = house": {"uc_as_house": True}}
    est = pd.concat({k: A.estimate(fe, final, sm["k"], k_f=k_f, published_es=PUBLISHED_ES, **kw)
                     for k, kw in runs.items()}, names=["run"])
    display(est.drop(columns="widened by").unstack("run").round(4))
    resid = A.residual_table(fe, final, est.loc["main"], q)
    display(resid)
    display(A.other_form_counts(labels, final, fe))
    display(A.agreement_with_final(labels, final, "Monica", fe))
    strata.to_csv(ACC / "strata.csv"); est.to_csv(ACC / "estimates.csv"); resid.to_csv(ACC / "residual.csv")
    json.dump({**A.meta(RELEASE, GROUND_TRUTH, SEED), **{f"{n}_sha256": m["frame_sha256"] for n, m in metas.items()},
               "interpreters": R2, "labelled": int(labelled.sum()), "notebook": "14_accuracy_assessment.ipynb"},
              open(ACC / "run_meta.json", "w"), indent=1, default=str)
else:
    print("estimates: need both interpreters' labels")

## What this shows

*Rewrite this cell from the outputs once labelling and adjudication are done.*

**State on 2026-11-10 (round 2 drawn; no round-2 label yet).**
- Round 1 (calibration 30, pilot 75) failed the gate: κ 0.58 (95% CI 0.37–0.77), house / not house the same on 83%; S4 worst (κ 0.36). Most disagreements were *unsure* against *house* or *gone*. Labels archived on the shared drive with `pilot1_record.json`.
- Revision 2: forced-choice labels, a *check later* tick, a narrow *can't tell: photo unreadable*, a pace warning. New calibration (30) and pilot (74) from the reserve (step C3), none seen in round 1; the main sample is unchanged.
- The frames reproduce map v1 exactly (S1 2,450, S2 327, S3 56, S4 594); the addendum and S9 are as drawn on 2026-11-06.
- No accuracy number comes from this notebook yet. Until it does, the only Task 1 accuracy numbers to quote are recall 80.5% and precision 88.2% against the USGS 2016/17 tracing (notebooks 11–12).